In [ ]:
from __future__ import annotations

from typing import Optional, Callable, List

import torch
from torch.optim import Optimizer
from torch.optim.lr_scheduler import _LRScheduler

In [ ]:
class PowerScheduler(_LRScheduler):
    def __init__(
        self,
        optimizer: Optimizer,
        batch_size: int,
        *,
        a: float = 4.6,
        b: float = -0.51,
        max_lr: float = 2e-2,
        warmup_tokens: int = 1_000_000_000,
        decay_tokens: int = 0,
        total_tokens: Optional[int] = None,
        decay_fn: Optional[Callable[[int, int, int], float]] = None,
    ) -> None:
        if batch_size <= 0:
            raise ValueError("batch_size must be positive.")
        if b >= 0.0:
            raise ValueError("Exponent b should be negative (got b >= 0).")
        if warmup_tokens < 0 or decay_tokens < 0:
            raise ValueError("warmup_tokens and decay_tokens must be ≥ 0.")
        if decay_tokens > 0 and total_tokens is None:
            raise ValueError("total_tokens and decay_fn must be provided if decay_tokens > 0.")

        if total_tokens is not None and decay_tokens > 0:
            self._decay_start: Optional[int] = total_tokens - decay_tokens
            if self._decay_start <= warmup_tokens:
                raise ValueError("Final-decay window overlaps warm-up. Adjust total_tokens or decay_tokens.")
        else:
            self._decay_start = None

        self.optimizer: Optimizer = optimizer
        self.batch_size: int = batch_size

        self.a: float = a
        self.b: float = b
        self.max_lr: float = max_lr
        self.warmup_tokens: int = warmup_tokens
        self.decay_tokens: int = decay_tokens
        self.total_tokens = total_tokens
        self.decay_fn = decay_fn

        self.tokens_trained: int = 0

    def step(self, num_tokens: int) -> float: # type: ignore[override]
        if num_tokens <= 0:
            raise ValueError("num_tokens must be positive.")
        self.tokens_trained += num_tokens

        if self.tokens_trained < self.warmup_tokens:
            lr_target = self._power_lr(self.warmup_tokens)
            lr = lr_target * (self.tokens_trained / self.warmup_tokens)
        elif self._decay_start is None or  self.tokens_trained <= self._decay_start:
            lr = self._power_lr(self.tokens_trained)
        else:
            if self.total_tokens is None:
                raise ValueError("total_tokens is required if decay_tokens > 0")
            if self.decay_fn is None:
                self.decay_fn = self.linear_decay_fn

            lr = self._power_lr(self._decay_start) * self.decay_fn(self.tokens_trained, self.total_tokens, self.decay_tokens)

        self._apply_lr(lr)
        return lr

    def _power_lr(self, tokens_trained: int) -> float:
        lr = self.a * self.batch_size * (tokens_trained ** self.b)
        return min(self.max_lr, lr)

    def _apply_lr(self, lr: float) -> None:
        for group in self.optimizer.param_groups:
            group["lr"] = lr

    def get_last_lr(self) -> List[float]:
        return [group["lr"] for group in self.optimizer.param_groups]

    @property
    def lr(self) -> float:
        return self.optimizer.param_groups[0]["lr"]

    @staticmethod
    def linear_decay_fn(tokens_trained: int, total_tokens: int, decay_tokens: int) -> float:
        decay_start = total_tokens - decay_tokens
        return max(1.0 - ((tokens_trained - decay_start) / decay_tokens), 0.0)

In [ ]:
# --------------------------------------------------------------------------- #
# Example usage
# --------------------------------------------------------------------------- #
if __name__ == "__main__":  # pragma: no cover
    import torch.nn as nn

    # Dummy model
    net = nn.Sequential(nn.Embedding(30_000, 512), nn.Linear(512, 30_000))
    opt = torch.optim.AdamW(net.parameters(), lr=1.0)  # lr will be overridden

    SEQ_LEN = 4096
    GLOBAL_BS = 1024

    scheduler = PowerScheduler(
        optimizer=opt,
        batch_size=GLOBAL_BS,
        a=4.0,
        b=-0.51,
        warmup_tokens=1_000_000_000,
        decay_tokens=100_000_000,
        max_lr=2e-2,
        total_tokens=3_000_000_000_000,
    )

    tokens_per_step = SEQ_LEN * GLOBAL_BS
    for step in range(5):
        # Simulate an optimiser step
        opt.zero_grad(set_to_none=True)
        opt.step()

        lr = scheduler.step(tokens_per_step)
        print(
            f"Step {step:>3d} │ tokens={scheduler.tokens_trained:,} │ lr={lr:.6e}"
        )
